# 04. ComprasNet 연구 분석용 전처리

원본을 보존한 채 필요한 컬럼만 읽고 영문 snake_case 스키마, 품질 플래그, 안전한 식별자를 만든다.
`Participantes.csv`는 청크 처리해 partitioned Parquet으로 저장한다. 설명 기반 `item_group`은
임시 품목군이며 SKU 확정값이 아니고, 동시출현은 All-or-Nothing의 증거가 아니다.

In [1]:
# 프로젝트 루트를 찾아 상대경로와 로컬 모듈 import를 일관되게 사용한다.
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾을 수 없습니다. 저장소 안에서 노트북을 실행하세요.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEED = 20260826
PROJECT_ROOT

WindowsPath('D:/CODE/proj2/5pl')

In [2]:
# 경로·패키지·공통 변환 함수를 준비하고 필수 원본 누락 시 중단한다.
import hashlib
import re
import unicodedata
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "comprasnet"
PROCESSED = PROJECT_ROOT / "data" / "processed" / "comprasnet"
paths = {"auctions": RAW_DIR / "Licitações.csv", "items": RAW_DIR / "Itens.csv", "participants": RAW_DIR / "Participantes.csv"}
missing = [str(path.relative_to(PROJECT_ROOT)) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("필수 ComprasNet 원본 파일이 없습니다: " + ", ".join(missing))
PROCESSED.mkdir(parents=True, exist_ok=True)
PARTICIPANTS_OUT = PROCESSED / "participants"
EVENTS_OUT = PROCESSED / "supplier_item_events"
PARTICIPANTS_OUT.mkdir(parents=True, exist_ok=True)
EVENTS_OUT.mkdir(parents=True, exist_ok=True)
READ = {"sep": ";", "encoding": "cp1252", "decimal": ",", "dtype": str, "low_memory": False}

def clean_code(series):
    return series.astype("string").str.replace(r"\D", "", regex=True).replace("", pd.NA)
def brazil_number(series):
    return pd.to_numeric(series.astype("string").str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")
def normalize_description(value):
    if pd.isna(value): return ""
    text = unicodedata.normalize("NFKD", str(value).upper()).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^A-Z0-9]+", " ", text).strip()
def stable_group(value):
    return "G_" + hashlib.sha1(value.encode("utf-8")).hexdigest()[:12] if value else pd.NA
def auction_id(frame):
    return ("A_" + clean_code(frame["ug_code"]).fillna("NA") + "_" + clean_code(frame["purchase_modality_code"]).fillna("NA")
            + "_" + clean_code(frame["purchase_number"]).fillna("NA") + "_" + clean_code(frame["process_number"]).fillna("NA"))

In [3]:
# 조달 건 파일에서 필요한 컬럼만 읽고 날짜·금액·상태 플래그와 auction_id를 만든다.
auction_columns = ["Número Licitação", "Código UG", "Código Modalidade Compra", "Número Processo", "Objeto",
                   "Situação Licitação", "UF", "Município", "Data Resultado Compra", "Data Abertura", "Valor Licitação"]
auction_names = {"Número Licitação": "purchase_number", "Código UG": "ug_code", "Código Modalidade Compra": "purchase_modality_code",
                 "Número Processo": "process_number", "Objeto": "object_text", "Situação Licitação": "auction_status", "UF": "state",
                 "Município": "municipality", "Data Resultado Compra": "result_date", "Data Abertura": "opening_date", "Valor Licitação": "auction_total_value"}
auctions = pd.read_csv(paths["auctions"], usecols=auction_columns, **READ).rename(columns=auction_names)
auctions["auction_id"] = auction_id(auctions)
for column in ["result_date", "opening_date"]:
    auctions[column] = pd.to_datetime(auctions[column], dayfirst=True, errors="coerce")
auctions["auction_total_value"] = brazil_number(auctions["auction_total_value"])
auctions["state"] = auctions["state"].astype("string").str.upper().str.strip()
auctions["period"] = auctions["result_date"].dt.to_period("M").astype("string")
auctions["flag_invalid_date"] = auctions["result_date"].isna()
auctions["flag_invalid_region"] = ~auctions["state"].fillna("").str.fullmatch(r"[A-Z]{2}")
auctions["flag_nonpositive_value"] = auctions["auction_total_value"].le(0) | auctions["auction_total_value"].isna()
valid_status = auctions["auction_status"].fillna("").str.upper().str.contains("PUBLIC|HOMOLOG|ENCERR|RESULT", regex=True)
auctions["flag_abnormal_status"] = ~valid_status
auctions["is_valid_analysis_sample"] = ~auctions[["flag_invalid_date", "flag_invalid_region", "flag_nonpositive_value"]].any(axis=1)
auctions = auctions.sort_values("result_date").drop_duplicates("auction_id", keep="last")
auctions.to_parquet(PROCESSED / "auctions.parquet", index=False)
display(auctions.head(3))
print("auctions:", len(auctions), "valid:", int(auctions["is_valid_analysis_sample"].sum()))

,purchase_number,ug_code,purchase_modality_code,process_number,object_text,auction_status,state,municipality,result_date,opening_date,auction_total_value,auction_id,period,flag_invalid_date,flag_invalid_region,flag_nonpositive_value,flag_abnormal_status,is_valid_analysis_sample
0,12012,170120,5,15528000001201224,Objeto: Pregão Eletrônico - Contratação de emp...,Publicado,RJ,CAMPOS DOS GOYTACAZES,2013-01-02,2012-12-27,28296.0,A_170120_5_12012_15528000001201224,2013-01,False,False,False,False,True
506,1022012,153063,5,34218/2012,Objeto: Prestação de serviços de manutenção pr...,Publicado,PA,BELEM,2013-01-02,2012-12-11,138000.0,A_153063_5_1022012_342182012,2013-01,False,False,False,False,True
164,62012,160009,5,2012006,Objeto: Pregão Eletrônico - Serviço comum espe...,Publicado,AM,MANAUS,2013-01-02,2012-12-27,40941.08,A_160009_5_62012_2012006,2013-01,False,False,False,False,True


auctions: 170096 valid: 142017


In [4]:
# 품목 파일을 정제하고 개별 행 ID와 설명 기반 임시 품목군을 분리한다.
item_columns = ["Número Licitação", "Código UG", "Código Modalidade Compra", "Número Processo", "Código Item Compra",
                "Descrição", "Quantidade Item", "Valor Item", "Código Vencedor"]
item_names = {"Número Licitação": "purchase_number", "Código UG": "ug_code", "Código Modalidade Compra": "purchase_modality_code",
              "Número Processo": "process_number", "Código Item Compra": "purchase_item_code", "Descrição": "item_description",
              "Quantidade Item": "item_quantity", "Valor Item": "item_total_value", "Código Vencedor": "winner_code"}
items = pd.read_csv(paths["items"], usecols=item_columns, **READ).rename(columns=item_names)
items["purchase_item_code"] = clean_code(items["purchase_item_code"])
items["winner_code"] = clean_code(items["winner_code"])
items["auction_id"] = auction_id(items)
items["item_id"] = "I_" + items["purchase_item_code"].fillna("MISSING")
items["item_description_normalized"] = items["item_description"].map(normalize_description)
items["item_group"] = items["item_description_normalized"].map(stable_group).astype("string")
extracted = items["purchase_item_code"].str.extract(r"(?P<procurement_year>(?:19|20)\d{2})(?P<item_sequence>\d{5})$")
items[["procurement_year", "item_sequence"]] = extracted
items["item_quantity"] = brazil_number(items["item_quantity"])
items["item_total_value"] = brazil_number(items["item_total_value"])
items["unit_price"] = items["item_total_value"].div(items["item_quantity"].where(items["item_quantity"].gt(0)))
auction_lookup = auctions[["auction_id", "result_date", "opening_date", "period", "state", "auction_status"]]
items = items.merge(auction_lookup, on="auction_id", how="left", validate="many_to_one")
items["flag_nonpositive_quantity"] = items["item_quantity"].le(0) | items["item_quantity"].isna()
items["flag_nonpositive_value"] = items["item_total_value"].le(0) | items["item_total_value"].isna()
items["flag_invalid_item_code"] = items["purchase_item_code"].isna() | extracted["procurement_year"].isna()
items["flag_unmatched_auction"] = items["result_date"].isna()
items["flag_duplicate"] = items.duplicated("purchase_item_code", keep="first")
items["is_valid_analysis_sample"] = ~items[["flag_nonpositive_quantity", "flag_nonpositive_value", "flag_invalid_item_code", "flag_unmatched_auction", "flag_duplicate"]].any(axis=1)
items.to_parquet(PROCESSED / "items.parquet", index=False)
bundles = (items.loc[items["is_valid_analysis_sample"], ["auction_id", "item_group"]].drop_duplicates()
           .groupby("auction_id")["item_group"].agg(list).rename("item_groups").reset_index())
bundles["item_group_count"] = bundles["item_groups"].str.len()
bundles.to_parquet(PROCESSED / "auction_item_bundles.parquet", index=False)
display(items.head(3))
print("items:", len(items), "valid:", int(items["is_valid_analysis_sample"].sum()))

,purchase_number,ug_code,purchase_modality_code,process_number,purchase_item_code,item_description,item_quantity,item_total_value,winner_code,auction_id,...,opening_date,period,state,auction_status,flag_nonpositive_quantity,flag_nonpositive_value,flag_invalid_item_code,flag_unmatched_auction,flag_duplicate,is_valid_analysis_sample
0,12012,170120,5,15528000001201224,1701200500001201200001,"INSTALACAO / MANUTENCAO - ELEVADORES, ESCADAS ...",1,28296.0,05379701000105,A_170120_5_12012_15528000001201224,...,2012-12-27,2013-01,RJ,Publicado,False,False,False,False,False,True
1,12012,510918,5,35274000865201282,5109180500001201200001,PRÓTESE MODULAR DESARTICULAÇÃO JOELHO.,1,8930.0,09232222000104,A_510918_5_12012_35274000865201282,...,2012-12-14,2013-01,RS,Evento de Alteração Publicad,False,False,False,False,False,True
2,12012,510918,5,35274000865201282,5109180500001201200002,PRÓTESE MODULAR AMPUTAÇÃO TRANSTIBIAL.,1,6279.0,09232222000104,A_510918_5_12012_35274000865201282,...,2012-12-14,2013-01,RS,Evento de Alteração Publicad,False,False,False,False,False,True


items: 1365244 valid: 1350915


In [5]:
# 참여자 2GB 파일을 청크로 읽어 공급자 이력과 품목 이벤트를 partitioned Parquet으로 저장한다.
participant_columns = ["Número Licitação", "Código UG", "Código Modalidade Compra", "Número Processo", "Código Item Compra",
                       "Código Participante", "Nome Participante", "Flag Vencedor"]
participant_names = {"Número Licitação": "purchase_number", "Código UG": "ug_code", "Código Modalidade Compra": "purchase_modality_code",
                     "Número Processo": "process_number", "Código Item Compra": "purchase_item_code", "Código Participante": "supplier_code",
                     "Nome Participante": "supplier_name", "Flag Vencedor": "winner_flag_raw"}
item_lookup = items[["purchase_item_code", "auction_id", "item_id", "item_group", "result_date", "period"]].drop_duplicates("purchase_item_code")
chunk_manifest = []
CHUNK_SIZE = 250_000
reader = pd.read_csv(paths["participants"], usecols=participant_columns, chunksize=CHUNK_SIZE, **READ)
for chunk_number, chunk in enumerate(reader):
    chunk = chunk.rename(columns=participant_names)
    chunk["purchase_item_code"] = clean_code(chunk["purchase_item_code"])
    chunk["supplier_id"] = clean_code(chunk["supplier_code"])
    chunk["auction_id_raw"] = auction_id(chunk)
    chunk["is_winner"] = chunk["winner_flag_raw"].astype("string").str.upper().str.strip().eq("SIM")
    chunk = chunk.merge(item_lookup, on="purchase_item_code", how="left", validate="many_to_one")
    chunk["auction_id"] = chunk["auction_id"].fillna(chunk["auction_id_raw"])
    chunk["period"] = chunk["period"].fillna("unknown").astype(str)
    chunk["flag_invalid_supplier"] = chunk["supplier_id"].isna()
    chunk["flag_unmatched_item"] = chunk["item_id"].isna()
    key_text = chunk[["auction_id", "purchase_item_code", "supplier_id"]].fillna("").astype(str).agg("|".join, axis=1)
    chunk["event_id"] = key_text.map(lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest())
    chunk["flag_duplicate_within_chunk"] = chunk.duplicated("event_id", keep="first")
    chunk["is_valid_analysis_sample"] = ~chunk[["flag_invalid_supplier", "flag_unmatched_item", "flag_duplicate_within_chunk"]].any(axis=1)
    participant_out = chunk[["event_id", "auction_id", "purchase_item_code", "item_id", "item_group", "supplier_id", "supplier_name",
                             "is_winner", "result_date", "period", "flag_invalid_supplier", "flag_unmatched_item",
                             "flag_duplicate_within_chunk", "is_valid_analysis_sample"]]
    event_out = participant_out.loc[participant_out["is_valid_analysis_sample"],
                                    ["event_id", "auction_id", "item_id", "item_group", "supplier_id", "is_winner", "result_date", "period"]]
    pq.write_to_dataset(pa.Table.from_pandas(participant_out, preserve_index=False), root_path=str(PARTICIPANTS_OUT),
                        partition_cols=["period"], basename_template=f"part-{chunk_number:05d}-{{i}}.parquet")
    pq.write_to_dataset(pa.Table.from_pandas(event_out, preserve_index=False), root_path=str(EVENTS_OUT),
                        partition_cols=["period"], basename_template=f"part-{chunk_number:05d}-{{i}}.parquet")
    chunk_manifest.append({"chunk": chunk_number, "rows": len(chunk), "valid_rows": len(event_out),
                           "unmatched_items": int(chunk["flag_unmatched_item"].sum()),
                           "duplicates_within_chunk": int(chunk["flag_duplicate_within_chunk"].sum())})
    if (chunk_number + 1) % 10 == 0:
        print(f"{chunk_number + 1}개 청크 처리 완료")
manifest = pd.DataFrame(chunk_manifest)
manifest.to_csv(PROCESSED / "preprocessing_manifest.csv", index=False, encoding="utf-8-sig")
display(manifest.tail())
print("전체 참여 행:", f"{manifest['rows'].sum():,}", "유효 이벤트:", f"{manifest['valid_rows'].sum():,}")

10개 청크 처리 완료


20개 청크 처리 완료


30개 청크 처리 완료


,chunk,rows,valid_rows,unmatched_items,duplicates_within_chunk
33,33,250000,222150,27850,4864
34,34,250000,225724,24276,4128
35,35,250000,224546,25454,4796
36,36,250000,225061,24939,3388
37,37,54097,48885,5212,268


전체 참여 행: 9,304,097 유효 이벤트: 8,410,284


## 산출물과 해석 주의

`auctions.parquet`, `items.parquet`, `participants/`, `supplier_item_events/`를 생성한다.
`Valor Item`은 총 품목금액이며 `unit_price = item_total_value / item_quantity`이다.
`item_id`는 조달 품목 행 ID, `item_group`은 설명 기반 탐색용 그룹이다. 전처리는 원본 행을
삭제하지 않고 플래그와 `is_valid_analysis_sample`을 제공한다. 청크 경계를 넘는 동일 이벤트는
`event_id`로 식별되며 분석 단계에서 전역 중복 제거한다.